In [1]:
import requests
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
class RussianOllamaClient:
    def __init__(self, model_name="llama3:latest"):
        self.model_name = model_name
        self.base_url = "http://localhost:11434"
        print(f"Используем модель: {model_name}")
    
    def make_request(self, prompt: str, max_tokens: int = 100, temperature: float = 0.3) -> str:
        data = {
            "model": self.model_name,
            "prompt": prompt,
            "stream": False,
            "options": {
                "temperature": temperature,
                "num_predict": max_tokens
            }
        }
        
        try:
            response = requests.post(
                f"{self.base_url}/api/generate", 
                json=data, 
                timeout=500
            )
            
            if response.status_code == 200:
                result = response.json()
                return result['response']
            else:
                print(f"❌ Ошибка {response.status_code}")
                return ""
                
        except Exception as e:
            print(f"❌ Исключение: {e}")
            return ""

In [29]:
# ollama serve
# & "C:\Users\zvrva\AppData\Local\Programs\Ollama\ollama.exe" serve

In [3]:
client = RussianOllamaClient("llama3:latest")  
    
test_response = client.make_request("Скажи 'тест пройден'", temperature=0.3)
if test_response:
    print("✅ Модель работает")
else:
    print("❌ Модель не отвечает")

Используем модель: llama3:latest
✅ Модель работает


In [4]:
try:
    df = pd.read_csv('ru_cefr_short.csv')
    print("✅ Файл найден:")
except:
    print("❌ Файл не найден, используем тестовые данные:")
    test_data = {
        'fragment': [
            "Весной, летом и осенью почти каждую субботу он играет в футбол. У них в школе есть футбольная команда. А зимой он играет в хоккей. Ещё мы любим театр.",
            "Вчера я весь вечер повторял грамматику, учил слова. В контрольной работе я сделал 3 ошибки. Питер и Кен написали всё без ошибок. Преподаватель сказал, что они делают большие успехи.",
            "В такой ситуации особенно сложно работающим студентам, которые должны находить время и для учёбы, и для работы. Это нередко вызывает стресс.",
            "Как редкое животное они стоили очень дорого и являлись предметом роскоши. Археологи нашли останки этих животных в развалинах домов богатых людей четвёртого века нашей эры.",
            "Он увлёкся энтомологией ещё мальчиком и в детстве познакомился с основными учёными трудами в этой области.",
            "Так же радиация — это частица, которая летит на огромной скорости. Сама частица может быть почти любой: от атомного ядра до электрона или фотона."
        ],
        'textbook-assigned cefr level': [1, 2, 3, 4, 5, 6]
    }
    df = pd.DataFrame(test_data)

df

✅ Файл найден:


,fragment,textbook-assigned cefr level
0,"Весной, летом и осенью почти каждую субботу он...",1
1,"Все говорят, что мама хорошая хозяйка. А ещё н...",1
2,На каждой двери красные плакаты и красные фона...,1
3,"Я считаю деньги, в час обедаю в кафе, а потом ...",1
4,Магазин «Чёрный квадрат» открывается в 9 часов...,1
...,...,...
7317,Утечка мозгов стала ключевым трендом междунаро...,6
7318,"По оценкам менеджеров «Промы», такая ситуация ...",6
7319,"Но это не мы, а техно-мемы заполоняют мир благ...",6
7320,Mapillary использует программное обеспечение д...,6


In [5]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['fragment'].values,
    df['textbook-assigned cefr level'].values,
    test_size=0.2,
    random_state=42,
    stratify=df['textbook-assigned cefr level']
)

len(train_texts)

5857

In [6]:
texts_to_augment = []

for i in range(len(train_labels)):
  if train_labels[i] == 6:
    texts_to_augment.append(train_texts[i])


len(texts_to_augment)

120

# 11. Перефразирование

In [7]:
def create_prompt(text):
    return f"""Перефразируй этот текст на русском языке, сохраняя его уровень сложности. Текст должен оставаться на том же уровне сложности.

Пример:

Оригинал: 'Искусственный интеллект меняет способы взаимодействия людей с технологиями.'

Аугментация: 'Технологии искусственного интеллекта трансформируют методы коммуникации человека с цифровыми устройствами.'

Текст: '{text}'

Верни ТОЛЬКО новый текст на русском языке без пояснений и комментариев, не заключай новый текст в кавычки:"""

## Temperature = 0.0

In [35]:
temperature=0.0


df_pred = pd.DataFrame(columns=['text', 'augmented-text'])

n = 1
print(f"Аугментируем {len(texts_to_augment)} текстов...")

for text in texts_to_augment:

    prompt = create_prompt(text)
    augmented_text = client.make_request(prompt, temperature)
    
    print(f"\n{n}. Реальный текст:\n\t{text}")
    print(f"Аугметированный текст:\n\t{augmented_text}")
   
    new_row = pd.DataFrame({
        'text': [text],
        'augmented-text': [augmented_text]
    })
    df_pred = pd.concat([df_pred, new_row], ignore_index=True)

    n+=1


df_pred.to_csv(f"c2_augmented_11_temp_{str(temperature).replace('.', '_')}.csv")

Аугментируем 120 текстов...

1. Реальный текст:
	3:38–3:54 Собственно, сама по себе радиация незаразна. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также лечат людей.
Аугметированный текст:
	3:38–3:54 В природе радиация как таковая не заразна. Например, мощные дозы излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также используются для лечения людей.

2. Реальный текст:
	Недавно компания Uber объявила об инвестиции одного миллиарда долларов в собственную систему автоматизированного вождения, а компания Waymo даже запустила приложение на Android для владельцев их самоуправляемых машин.
Аугметированный текст:
	Компания Uber объявила о вложении миллиарда долларов в собственное решение по автоматизации управления, а компания Waymo уже запустила приложение для владельцев своих самоуправляемых транспортных средств на платформе Android.

3. Реальный текст:
	Множество повестей: «Двойник», «Дядюшкин сон», «Записки из подп

## Temperature = 0.1

In [8]:
temperature=0.1


df_pred = pd.DataFrame(columns=['text', 'augmented-text'])

n = 1
print(f"Аугментируем {len(texts_to_augment)} текстов...")

for text in texts_to_augment:

    prompt = create_prompt(text)
    augmented_text = client.make_request(prompt, temperature)
    
    print(f"\n{n}. Реальный текст:\n\t{text}")
    print(f"Аугметированный текст:\n\t{augmented_text}")
   
    new_row = pd.DataFrame({
        'text': [text],
        'augmented-text': [augmented_text]
    })
    df_pred = pd.concat([df_pred, new_row], ignore_index=True)

    n+=1


df_pred.to_csv(f"c2_augmented_11_temp_{str(temperature).replace('.', '_')}.csv")

Аугментируем 120 текстов...

1. Реальный текст:
	3:38–3:54 Собственно, сама по себе радиация незаразна. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также лечат людей.
Аугметированный текст:
	4:38–4:54 В самом себе радиация не заразна. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также лечат людей.

2. Реальный текст:
	Недавно компания Uber объявила об инвестиции одного миллиарда долларов в собственную систему автоматизированного вождения, а компания Waymo даже запустила приложение на Android для владельцев их самоуправляемых машин.
Аугметированный текст:
	Компания Uber объявила о вливании миллиарда долларов в разработку собственной системы автоматизированного управления транспортными средствами, а компания Waymo уже запустила приложение для владельцев своих самоуправляющихся машин на платформе Android.

3. Реальный текст:
	Множество повестей: «Двойник», «Дядюшкин сон», «Записки из под

## Temperature = 0.2

In [9]:
temperature=0.2


df_pred = pd.DataFrame(columns=['text', 'augmented-text'])

n = 1
print(f"Аугментируем {len(texts_to_augment)} текстов...")

for text in texts_to_augment:

    prompt = create_prompt(text)
    augmented_text = client.make_request(prompt, temperature)
    
    print(f"\n{n}. Реальный текст:\n\t{text}")
    print(f"Аугметированный текст:\n\t{augmented_text}")
   
    new_row = pd.DataFrame({
        'text': [text],
        'augmented-text': [augmented_text]
    })
    df_pred = pd.concat([df_pred, new_row], ignore_index=True)

    n+=1


df_pred.to_csv(f"c2_augmented_11_temp_{str(temperature).replace('.', '_')}.csv")

Аугментируем 120 текстов...

1. Реальный текст:
	3:38–3:54 Собственно, сама по себе радиация незаразна. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также лечат людей.
Аугметированный текст:
	4:38–4:54 В природе сама радиация не заразна. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также используются для лечения людей.

2. Реальный текст:
	Недавно компания Uber объявила об инвестиции одного миллиарда долларов в собственную систему автоматизированного вождения, а компания Waymo даже запустила приложение на Android для владельцев их самоуправляемых машин.
Аугметированный текст:
	Компания Uber объявила о вложении миллиарда долларов в собственную систему автоматизированного управления транспортными средствами, а компания Waymo уже запустила приложение для владельцев их самоуправляющихся машин на платформе Android.

3. Реальный текст:
	Множество повестей: «Двойник», «Дядюшкин сон», «Записки

## Temperature = 0.4

In [8]:
temperature=0.4


df_pred = pd.DataFrame(columns=['text', 'augmented-text'])

n = 1
print(f"Аугментируем {len(texts_to_augment)} текстов...")

for text in texts_to_augment:

    prompt = create_prompt(text)
    augmented_text = client.make_request(prompt, temperature)
    
    print(f"\n{n}. Реальный текст:\n\t{text}")
    print(f"Аугметированный текст:\n\t{augmented_text}")
   
    new_row = pd.DataFrame({
        'text': [text],
        'augmented-text': [augmented_text]
    })
    df_pred = pd.concat([df_pred, new_row], ignore_index=True)

    n+=1


df_pred.to_csv(f"c2_augmented_11_temp_{str(temperature).replace('.', '_')}.csv")

Аугментируем 120 текстов...

1. Реальный текст:
	3:38–3:54 Собственно, сама по себе радиация незаразна. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также лечат людей.
Аугметированный текст:
	5:38–5:54 В природе самой радиация не является заразной. Например, мощными дозами излучения от тех же изотопов часто стерилизуют фрукты и овощи, а также используются в медицине для лечения людей.

2. Реальный текст:
	Недавно компания Uber объявила об инвестиции одного миллиарда долларов в собственную систему автоматизированного вождения, а компания Waymo даже запустила приложение на Android для владельцев их самоуправляемых машин.
Аугметированный текст:
	Компания Uber объявила о вложении миллиарда долларов в собственную систему автоматизированного управления транспортными средствами, а компания Waymo уже запустила приложение для владельцев их самоуправляемых машин на платформе Android.

3. Реальный текст:
	Множество повестей: «Двойник», «Дядюшкин 

## Temperature = 0.5

In [9]:
temperature=0.5


df_pred = pd.DataFrame(columns=['text', 'augmented-text'])

n = 1
print(f"Аугментируем {len(texts_to_augment)} текстов...")

for text in texts_to_augment:

    prompt = create_prompt(text)
    augmented_text = client.make_request(prompt, temperature)
    
    print(f"\n{n}. Реальный текст:\n\t{text}")
    print(f"Аугметированный текст:\n\t{augmented_text}")
   
    new_row = pd.DataFrame({
        'text': [text],
        'augmented-text': [augmented_text]
    })
    df_pred = pd.concat([df_pred, new_row], ignore_index=True)

    n+=1


df_pred.to_csv(f"c2_augmented_11_temp_{str(temperature).replace('.', '_')}.csv")

Аугментируем 120 текстов...

1. Реальный текст:
	3:38–3:54 Собственно, сама по себе радиация незаразна. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также лечат людей.
Аугметированный текст:
	5:38–5:54 В природе радиация как таковая не заражает. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также используются для лечения людей.

2. Реальный текст:
	Недавно компания Uber объявила об инвестиции одного миллиарда долларов в собственную систему автоматизированного вождения, а компания Waymo даже запустила приложение на Android для владельцев их самоуправляемых машин.
Аугметированный текст:
	Компания Uber объявила о вложении миллиарда долларов в собственную систему автоматизированного управления транспортными средствами, а компания Waymo запустила приложение для владельцев своих самоуправляющихся машин на платформе Android.

3. Реальный текст:
	Множество повестей: «Двойник», «Дядюшкин сон», «

## Temperature = 0.6

In [10]:
temperature=0.6


df_pred = pd.DataFrame(columns=['text', 'augmented-text'])

n = 1
print(f"Аугментируем {len(texts_to_augment)} текстов...")

for text in texts_to_augment:

    prompt = create_prompt(text)
    augmented_text = client.make_request(prompt, temperature)
    
    print(f"\n{n}. Реальный текст:\n\t{text}")
    print(f"Аугметированный текст:\n\t{augmented_text}")
   
    new_row = pd.DataFrame({
        'text': [text],
        'augmented-text': [augmented_text]
    })
    df_pred = pd.concat([df_pred, new_row], ignore_index=True)

    n+=1


df_pred.to_csv(f"c2_augmented_11_temp_{str(temperature).replace('.', '_')}.csv")

Аугментируем 120 текстов...

1. Реальный текст:
	3:38–3:54 Собственно, сама по себе радиация незаразна. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также лечат людей.
Аугметированный текст:
	Самая по себе радиация не заражает. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также лечат людей.

2. Реальный текст:
	Недавно компания Uber объявила об инвестиции одного миллиарда долларов в собственную систему автоматизированного вождения, а компания Waymo даже запустила приложение на Android для владельцев их самоуправляемых машин.
Аугметированный текст:
	Компания Uber объявила о вложении миллиарда долларов в разработку собственной системы автоматизированного управления транспортными средствами, а компания Waymo уже запустила приложение для владельцев их самоуправляемых машин на платформе Android.

3. Реальный текст:
	Множество повестей: «Двойник», «Дядюшкин сон», «Записки из подполья» и друг

## Temperature = 0.7

In [11]:
temperature=0.7


df_pred = pd.DataFrame(columns=['text', 'augmented-text'])

n = 1
print(f"Аугментируем {len(texts_to_augment)} текстов...")

for text in texts_to_augment:

    prompt = create_prompt(text)
    augmented_text = client.make_request(prompt, temperature)
    
    print(f"\n{n}. Реальный текст:\n\t{text}")
    print(f"Аугметированный текст:\n\t{augmented_text}")
   
    new_row = pd.DataFrame({
        'text': [text],
        'augmented-text': [augmented_text]
    })
    df_pred = pd.concat([df_pred, new_row], ignore_index=True)

    n+=1


df_pred.to_csv(f"c2_augmented_11_temp_{str(temperature).replace('.', '_')}.csv")

Аугментируем 120 текстов...

1. Реальный текст:
	3:38–3:54 Собственно, сама по себе радиация незаразна. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также лечат людей.
Аугметированный текст:
	4:38–4:54 В природе сама радиация не заразна. Например, высокими дозами излучения от тех же изотопов часто стерилизуют фрукты и овощи, а также используются для лечения людей.

2. Реальный текст:
	Недавно компания Uber объявила об инвестиции одного миллиарда долларов в собственную систему автоматизированного вождения, а компания Waymo даже запустила приложение на Android для владельцев их самоуправляемых машин.
Аугметированный текст:
	Компания Uber объявила о вложении одного миллиарда долларов в собственную систему автоматизированного управления транспортными средствами, а компания Waymo уже запустила приложение для владельцев их самоуправляемых машин на платформе Android.

3. Реальный текст:
	Множество повестей: «Двойник», «Дядюшкин сон», «Записки

## Temperature = 0.8

In [12]:
temperature=0.8


df_pred = pd.DataFrame(columns=['text', 'augmented-text'])

n = 1
print(f"Аугментируем {len(texts_to_augment)} текстов...")

for text in texts_to_augment:

    prompt = create_prompt(text)
    augmented_text = client.make_request(prompt, temperature)
    
    print(f"\n{n}. Реальный текст:\n\t{text}")
    print(f"Аугметированный текст:\n\t{augmented_text}")
   
    new_row = pd.DataFrame({
        'text': [text],
        'augmented-text': [augmented_text]
    })
    df_pred = pd.concat([df_pred, new_row], ignore_index=True)

    n+=1


df_pred.to_csv(f"c2_augmented_11_temp_{str(temperature).replace('.', '_')}.csv")

Аугментируем 120 текстов...

1. Реальный текст:
	3:38–3:54 Собственно, сама по себе радиация незаразна. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также лечат людей.
Аугметированный текст:
	4:38–4:54 В природе самой радиация не является заразной. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также используются для лечения людей.

2. Реальный текст:
	Недавно компания Uber объявила об инвестиции одного миллиарда долларов в собственную систему автоматизированного вождения, а компания Waymo даже запустила приложение на Android для владельцев их самоуправляемых машин.
Аугметированный текст:
	Компания Uber объявила о вложении в миллиард долларов в собственную систему автоматизированного управления транспортными средствами, а компания Waymo уже запустила приложение для владельцев самоуправляемых машин на платформе Android.

3. Реальный текст:
	Множество повестей: «Двойник», «Дядюшкин сон», «

## Temperature = 0.9

In [13]:
temperature=0.9


df_pred = pd.DataFrame(columns=['text', 'augmented-text'])

n = 1
print(f"Аугментируем {len(texts_to_augment)} текстов...")

for text in texts_to_augment:

    prompt = create_prompt(text)
    augmented_text = client.make_request(prompt, temperature)
    
    print(f"\n{n}. Реальный текст:\n\t{text}")
    print(f"Аугметированный текст:\n\t{augmented_text}")
   
    new_row = pd.DataFrame({
        'text': [text],
        'augmented-text': [augmented_text]
    })
    df_pred = pd.concat([df_pred, new_row], ignore_index=True)

    n+=1


df_pred.to_csv(f"c2_augmented_11_temp_{str(temperature).replace('.', '_')}.csv")

Аугментируем 120 текстов...

1. Реальный текст:
	3:38–3:54 Собственно, сама по себе радиация незаразна. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также лечат людей.
Аугметированный текст:
	4:38–4:54 В природе радиация сама по себе не заражает. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также используются для лечения людей.

2. Реальный текст:
	Недавно компания Uber объявила об инвестиции одного миллиарда долларов в собственную систему автоматизированного вождения, а компания Waymo даже запустила приложение на Android для владельцев их самоуправляемых машин.
Аугметированный текст:
	Компания Uber объявила о вложении одного миллиарда долларов в собственную систему автоматизированного управления транспортными средствами, а компания Waymo уже запустила приложение для владельцев их самоуправляемых машин на платформе Android.

3. Реальный текст:
	Множество повестей: «Двойник», «Дядюшкин 

# Temperature=1.0

In [14]:
temperature=1.0


df_pred = pd.DataFrame(columns=['text', 'augmented-text'])

n = 1
print(f"Аугментируем {len(texts_to_augment)} текстов...")

for text in texts_to_augment:

    prompt = create_prompt(text)
    augmented_text = client.make_request(prompt, temperature)
    
    print(f"\n{n}. Реальный текст:\n\t{text}")
    print(f"Аугметированный текст:\n\t{augmented_text}")
   
    new_row = pd.DataFrame({
        'text': [text],
        'augmented-text': [augmented_text]
    })
    df_pred = pd.concat([df_pred, new_row], ignore_index=True)

    n+=1


df_pred.to_csv(f"c2_augmented_11_temp_{str(temperature).replace('.', '_')}.csv")

Аугментируем 120 текстов...

1. Реальный текст:
	3:38–3:54 Собственно, сама по себе радиация незаразна. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также лечат людей.
Аугметированный текст:
	С

2. Реальный текст:
	Недавно компания Uber объявила об инвестиции одного миллиарда долларов в собственную систему автоматизированного вождения, а компания Waymo даже запустила приложение на Android для владельцев их самоуправляемых машин.
Аугметированный текст:
	Ком

3. Реальный текст:
	Множество повестей: «Двойник», «Дядюшкин сон», «Записки из подполья» и другие, рассказы, публицистика, дневники, даже стихи были в творчестве великого автора.
Аугметированный текст:
	М

4. Реальный текст:
	Встречи одноклассников и одногруппников превращаются в вечеринки понтов, одни рвутся туда – доложить о своей крутизне, другие прячутся на илистом дне – нечем козырнуть. Жить мнением других.
Аугметированный текст:
	В

5. Реальный текст:
	Самопрезентация — как гл

# Temperature=1.1

In [8]:
temperature=1.1


df_pred = pd.DataFrame(columns=['text', 'augmented-text'])

n = 1
print(f"Аугментируем {len(texts_to_augment)} текстов...")

for text in texts_to_augment:

    prompt = create_prompt(text)
    augmented_text = client.make_request(prompt, temperature)
    
    print(f"\n{n}. Реальный текст:\n\t{text}")
    print(f"Аугметированный текст:\n\t{augmented_text}")
   
    new_row = pd.DataFrame({
        'text': [text],
        'augmented-text': [augmented_text]
    })
    df_pred = pd.concat([df_pred, new_row], ignore_index=True)

    n+=1


df_pred.to_csv(f"c2_augmented_11_temp_{str(temperature).replace('.', '_')}.csv")

Аугментируем 120 текстов...

1. Реальный текст:
	3:38–3:54 Собственно, сама по себе радиация незаразна. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также лечат людей.
Аугметированный текст:
	С

2. Реальный текст:
	Недавно компания Uber объявила об инвестиции одного миллиарда долларов в собственную систему автоматизированного вождения, а компания Waymo даже запустила приложение на Android для владельцев их самоуправляемых машин.
Аугметированный текст:
	Ком

3. Реальный текст:
	Множество повестей: «Двойник», «Дядюшкин сон», «Записки из подполья» и другие, рассказы, публицистика, дневники, даже стихи были в творчестве великого автора.
Аугметированный текст:
	М

4. Реальный текст:
	Встречи одноклассников и одногруппников превращаются в вечеринки понтов, одни рвутся туда – доложить о своей крутизне, другие прячутся на илистом дне – нечем козырнуть. Жить мнением других.
Аугметированный текст:
	В

5. Реальный текст:
	Самопрезентация — как гл